# Advanced Binary Classification Pipeline

Dataset: **Bank Personal Loan Modelling**  
Target: **Personal Loan**  
Primary model family: **Logistic Regression** (advanced variants only)

This notebook demonstrates:
- advanced feature engineering
- multicollinearity handling (correlation + VIF)
- hyperparameter tuning
- threshold optimization (F1 and Youden's J)
- class imbalance handling (`class_weight`, `SMOTE`)
- calibration
- comparison across Logistic Regression variants only

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from src.logistic_regression_loan import main

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 1. Run Full Training Pipeline

In [ ]:
# This executes all LR training variants, tuning, threshold optimization,
# calibration, and saves figures/output tables.
main()

## 2. Load Key Output Tables

In [ ]:
metrics_df = pd.read_csv('outputs/all_model_metrics.csv')
threshold_df = pd.read_csv('outputs/threshold_analysis.csv')
vif_df = pd.read_csv('outputs/final_vif_table.csv')

metrics_df

## 3. Best Model and Threshold Summary

In [ ]:
with open('outputs/advanced_run_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

summary

In [ ]:
best_model_row = metrics_df.iloc[0]
best_model_row

## 4. Threshold Optimization Table (Top Rows by F1)

In [ ]:
threshold_df.sort_values(['f1', 'recall'], ascending=False).head(15)

## 5. VIF Table (After Feature Filtering)

In [ ]:
vif_df.sort_values('vif', ascending=False)

## 6. Display Generated Plots

In [ ]:
plot_files = [
    'figures/confusion_matrices_lr_variants.png',
    'figures/roc_curves_lr_variants.png',
    'figures/pr_curves_lr_variants.png',
    'figures/threshold_vs_f1_youden.png',
    'figures/calibration_curves.png',
    'figures/lr_feature_importance.png',
]

for p in plot_files:
    img = plt.imread(p)
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(Path(p).name)
    plt.show()

## 7. Focused Logistic Regression Comparison

In [ ]:
lr_only = metrics_df[metrics_df['model'].str.contains('LR', case=False, na=False)]
lr_only

## 8. Report-Ready Conclusions

Suggested interpretation pattern:
1. Tuned LR improved recall over baseline while keeping strong ROC-AUC.
2. Threshold tuning provided better precision-recall control than fixed 0.50.
3. Class-imbalance methods (`balanced`, `SMOTE`) increased recall but reduced precision.
4. Calibration improved probability reliability (see calibration curves).
5. Calibrated LR and tuned-threshold LR provide the strongest LR trade-offs depending on whether you prioritize F1 or recall.